# BudgetPylot — Architecture POO en Python
## De la modélisation financière à l'application web Flask

> **Objectif de ce notebook** : illustrer comment une application financière réelle
> est construite en **Programmation orientée objet (POO)** en Python.
> Chaque classe modélise un item que chaque personne ayant un compte bancaire peut avoir (compte, revenu, charge, crédit, épargne, patrimoine), ainsi que les relations entre classes. Grâce aux données entrées dans chaque objet, on va ensuite pouvoir faire des statistiques avec des méthodes de calculs bancaires.


## 1. Pourquoi la POO pour une application financière ?

La POO permet de **modéliser le monde réel** directement dans le code.
En finance personnelle, nous avons par exemple :

| Concept réel | Classe Python |
|---|---|
| Une personne | `Client` |
| Un compte bancaire | `Compte` |
| Un salaire | `Revenu` |
| Un loyer mensuel | `Depense` |
| Un prêt immobilier | `Credit` |
| Un Livret A | `Epargne` |
| Un appartement | `Patrimoine` |

Chaque objet contient :
- des **données** (ex : montant, date)
- des **règles** (ex : calcul mensuel, plafonds)

### Pourquoi c’est utile ?

- **Logique claire** : chaque classe gère son propre rôle
- **Code réutilisable** : un même `Credit` peut être partagé entre plusieurs `Client`
- **Facile à faire évoluer** : ajouter un type de revenu ou de dépense est simple

Résultat : un code plus lisible et structuré

## 2. Les classes de base — Modélisation des items client (revenus, dépenses)

In [11]:
from datetime import datetime
from typing import List
import calendar

# REVENU 
class Revenu:
    """
    Représente un revenu d'un client.
    """
    
    TYPE_REVENU = (
        "Salaire", "Prime fixe", "Bonus", "Revenu non salarié", "Allocations chômage",
        "Pension d'invalidité", "Retraite", "RSA", "APL", "Allocations familiales",
        "Prime d'activité", "Bourse étudiante", "Revenu locatif", "Revenu SCPI", "Intérêt d'épargne",
        "Coupon obligataire", "Héritage", "Donation", "Gain exceptionnel",
        "Vente d'actifs", "Indemnité", "Virement reçu", "Remboursement", "Autre"
    )
    PERIODE = ("mensuelle", "annuelle", "unique")

    def __init__(self, type_de_revenu: str, montant: float, periodicite: str,
                 jour: int = 1, mois: int | None = None):
        if type_de_revenu not in self.TYPE_REVENU:
            raise ValueError(f"Type invalide. Valeurs acceptées : {self.TYPE_REVENU}")
        if periodicite not in self.PERIODE:
            raise ValueError(f"Périodicité invalide.")
        if montant <= 0:
            raise ValueError("Le montant doit être positif")
        if periodicite in ("annuelle", "unique") and mois is None:
            raise ValueError("Un revenu annuel ou unique nécessite un mois")
        
        self.type_de_revenu = type_de_revenu
        self.montant = float(montant)
        self.periodicite = periodicite
        self.jour = jour
        self.mois = mois
        self.est_revenu_principal = False

    def montant_pour_mois(self, mois_actuel: int) -> float:
        """Retourne le montant applicable pour un mois donné."""
        if self.periodicite == "mensuelle":
            return self.montant
        elif self.periodicite in ("annuelle", "unique") and self.mois == mois_actuel:
            return self.montant
        return 0.0

    def __repr__(self):
        return f"Revenu({self.type_de_revenu}, {self.montant} EUR, {self.periodicite})"


# DEPENSE
class Depense:
    """
    Modélise une dépense, sa temporalité (fixe/variable/unique) et sa catégorie.
    La fonction est_a_appliquer() détermine si la dépense est due un mois donné.
    """
    
    TYPE_DEPENSE = ("fixe", "variable_mensuelle", "unique")
    CATEGORIE_DEPENSE = (
        "Loyer", "Assurance habitation", "Assurance automobile",
        "Electricité", "Alimentation", "Transport", "Impôts et taxes",
        "Loisir", "Autre (hors crédit, LLD, LOA)"
    )

    def __init__(self, nom: str, montant: float, type_depense: str,
                 categorie_depense: str, jour: int = 1, mois: int | None = None):
        if type_depense not in self.TYPE_DEPENSE:
            raise ValueError("Type de dépense invalide")
        if categorie_depense not in self.CATEGORIE_DEPENSE:
            raise ValueError("Catégorie invalide")
        if montant <= 0:
            raise ValueError("Montant positif requis")
        if type_depense == "unique" and mois is None:
            raise ValueError("Une dépense unique nécessite un mois")
        
        self.nom = nom
        self.montant = float(montant)
        self.type_depense = type_depense
        self.categorie_depense = categorie_depense
        self.jour = jour
        self.mois = mois

    def est_a_appliquer(self, mois_actuel: int) -> bool:
        """Vérifie si cette dépense est applicable pour le mois donné."""
        if self.type_depense in ("fixe", "variable_mensuelle"):
            return True
        return self.type_depense == "unique" and self.mois == mois_actuel

    def __repr__(self):
        return f"Depense({self.nom}, {self.montant} EUR, {self.type_depense})"



# TEST
salaire = Revenu("Salaire", 3500, "mensuelle", jour=28)
prime = Revenu("Prime fixe", 6000, "annuelle", mois=12)
loyer = Depense("Loyer appartement", 1100, "fixe", "Loyer", jour=1)
vacances = Depense("Vacances été", 2500, "unique", "Loisir", jour=15, mois=8)

print("=== Revenus ===")
print(salaire)
print(prime)
print(f"Salaire en mars : {salaire.montant_pour_mois(3)} €")
print(f"Prime en mars : {prime.montant_pour_mois(3)} €")
print(f"Prime en déc : {prime.montant_pour_mois(12)} €")

print()
print("=== Dépenses ===")
print(loyer)
print(f"Loyer applicable en mars: {loyer.est_a_appliquer(3)}")
print(f"Vacances en mars : {vacances.est_a_appliquer(3)}")
print(f"Vacances en août: {vacances.est_a_appliquer(8)}")


=== Revenus ===
Revenu(Salaire, 3500.0 EUR, mensuelle)
Revenu(Prime fixe, 6000.0 EUR, annuelle)
Salaire en mars : 3500.0 €
Prime en mars : 0.0 €
Prime en déc : 6000.0 €

=== Dépenses ===
Depense(Loyer appartement, 1100.0 EUR, fixe)
Loyer applicable en mars: True
Vacances en mars : False
Vacances en août: True


## 3. La classe Credit — Gestion des co-emprunteurs

Un `Credit` peut avoir **plusieurs emprunteurs avec des parts différentes**.
C'est un exemple classique que l'on peut voir dans la vie réelle.

La pondération est importante: si Alban détient 50% d'un crédit de 1 000 €/mois,
sa charge est de **500 EUR**, pas 1 000 EUR.


In [4]:
class Credit:
    """
    Modélise un crédit avec gestion des co-emprunteurs et de leurs parts.
    
    Règle importante : la somme des parts ne peut pas dépasser 100%.
    """

    REGLES_CREDIT = {
        1: {"nom": "Crédit immobilier amortissable", "duree_max": 360},
        2: {"nom": "Crédit in fine",  "duree_max": 360},
        3: {"nom": "Prêt relais", "duree_max": 36},
        4: {"nom": "Crédit à la consommation", "duree_max": 120},
        5: {"nom": "Crédit renouvelable", "duree_max": None},
        6: {"nom": "LOA","duree_max": None},
    }

    def __init__(self, type_de_credit: int, mensualite: float, jour_echeance: int,
                 crd: float | None = None, taux: float | None = None,
                 fin_credit=None):
        if type_de_credit not in self.REGLES_CREDIT:
            raise ValueError("Type de crédit invalide")
        if mensualite <= 0:
            raise ValueError("Mensualité positive requise")
        
        self.type_de_credit = type_de_credit
        self.mensualite = float(mensualite)
        self.jour_echeance = jour_echeance
        self.crd = float(crd) if crd is not None else None
        self.taux = float(taux) if taux is not None else None
        self.fin_credit = fin_credit
        self.emprunteur: dict = {} # {client: part}

    @property
    def nom(self) -> str:
        return self.REGLES_CREDIT[self.type_de_credit]["nom"]

    def ajouter_emprunteur(self, client, part: float = 1.0):
        """Enregistre un emprunteur avec sa part (0 < part <= 1)."""
        if not (0 < part <= 1):
            raise ValueError("La part doit être entre 0 et 1")
        total_existant = sum(p for c, p in self.emprunteur.items() if c is not client)
        if total_existant + part > 1.0001:
            raise ValueError(f"Parts totales dépassent 100% ({(total_existant + part)*100:.1f}%)")
        self.emprunteur[client] = part

    def mensualite_client(self, client) -> float:
        """Retourne la mensualité pondérée par la part du client."""
        if client not in self.emprunteur:
            raise ValueError("Ce client n'est pas emprunteur sur ce crédit")
        return self.mensualite * self.emprunteur[client]

    def __repr__(self):
        emprunteurs = ", ".join(f"{c.nom} ({p*100:.0f}%)" for c, p in self.emprunteur.items())
        return f"Credit({self.nom}, {self.mensualite} EUR/mois | {emprunteurs})"


# Simulation d'un crédit immobilier en indivision
# On crée des clients simplifiés pour la démo
class ClientSimple:
    def __init__(self, nom): self.nom = nom

hugo = ClientSimple("Hugo")
alban = ClientSimple("Alban")
copine_absente = ClientSimple("Copine (hors analyse)")

# Crédit locatif Hugo + Alban à 50/50
credit_locatif = Credit(
    type_de_credit=1,
    mensualite=920,
    jour_echeance=1,
    crd=145000,
    taux=3.5
)
credit_locatif.ajouter_emprunteur(hugo, 0.5)
credit_locatif.ajouter_emprunteur(alban, 0.5)

print("=== Crédit locatif Hugo + Alban ===")
print(credit_locatif)
print(f"Mensualité Hugo: {credit_locatif.mensualite_client(hugo):.0f} €")
print(f"Mensualité Alban : {credit_locatif.mensualite_client(alban):.0f} €")
print(f"Total vérifié: {credit_locatif.mensualite_client(hugo) + credit_locatif.mensualite_client(alban):.0f} €")

print()
# Crédit d'Antoine à 50% avec une copine hors analyse
credit_antoine_indivision = Credit(
    type_de_credit=1,
    mensualite=800,
    jour_echeance=1,
    crd=110000,
    taux=2.9
)
credit_antoine_indivision.ajouter_emprunteur(ClientSimple("Antoine"), 0.5)
credit_antoine_indivision.ajouter_emprunteur(copine_absente, 0.5)

antoine = list(credit_antoine_indivision.emprunteur.keys())[0]
print("=== Crédit Antoine (50% en indivision, sa copine est hors analyse) ===")
print(credit_antoine_indivision)
print(f"Charge d'Antoine : {credit_antoine_indivision.mensualite_client(antoine):.0f} EUR")
print(f"Correct : 800 × 50% = 400 EUR, pas 800 EUR")

print()
# Test de validation : parts > 100%
try:
    credit_test = Credit(1, 500, 1)
    credit_test.ajouter_emprunteur(ClientSimple("A"), 0.6)
    credit_test.ajouter_emprunteur(ClientSimple("B"), 0.6)  # <- doit lever une erreur
except ValueError as e:
    print(f"Validation OK — Erreur attendue : {e}")


=== Crédit locatif Hugo + Alban ===
Credit(Crédit immobilier amortissable, 920.0 EUR/mois | Hugo (50%), Alban (50%))
Mensualité Hugo: 460 €
Mensualité Alban : 460 €
Total vérifié: 920 €

=== Crédit Antoine (50% en indivision, sa copine est hors analyse) ===
Credit(Crédit immobilier amortissable, 800.0 EUR/mois | Antoine (50%), Copine (hors analyse) (50%))
Charge d'Antoine : 400 EUR
Correct : 800 × 50% = 400 EUR, pas 800 EUR

Validation OK — Erreur attendue : Parts totales dépassent 100% (120.0%)


## 4. La classe Epargne — Plafonds réglementaires et projections

La classe `Epargne` regroupe les règles réglementaires françaises (plafonds Livret A,
PEA, etc.) et permet des **projections avec intérêts composés** en tenant compte
des versements permanents ET ponctuels.

**Formule intérêts composés :**
```
S(n) = S(0) × (1 + t/12)^n + versement × [(1 + t/12)^n - 1] / (t/12)
```


In [3]:
class Epargne:
    """
    Modélise un produit d'épargne avec ses contraintes réglementaires.
    """

    TYPE_EPARGNE = {
        1:  {"nom": "Livret A", "plafond_min": 10,  "plafond_max": 22950},
        2:  {"nom": "LDDS", "plafond_min": 10,  "plafond_max": 12000},
        6:  {"nom": "PEL", "plafond_min": 225, "plafond_max": 61200},
        9:  {"nom": "Assurance Vie", "plafond_min": None,"plafond_max": None},
        14: {"nom": "PEA", "plafond_min": 15,  "plafond_max": 150000},
        16: {"nom": "Compte titres", "plafond_min": None,"plafond_max": None},
    }

    def __init__(self, type_epargne: int, solde: float,
                 versements_permanents: float = 0.0, taux: float = 0.0):
        if type_epargne not in self.TYPE_EPARGNE:
            raise ValueError("Type d'épargne invalide")
        if solde < 0:
            raise ValueError("Solde négatif interdit")
        
        produit = self.TYPE_EPARGNE[type_epargne]
        
        if produit["plafond_min"] and solde < produit["plafond_min"]:
            raise ValueError(f"Solde sous le minimum requis ({produit['plafond_min']} EUR)")
        if produit["plafond_max"] and solde > produit["plafond_max"]:
            raise ValueError(f"Plafond dépassé ({produit['plafond_max']} EUR)")
        
        self.type_epargne = type_epargne
        self.nom = produit["nom"]
        self.solde = float(solde)
        self.versements_permanents = float(versements_permanents)
        self.taux = float(taux)
        self.plafond_max = produit["plafond_max"]

    def projeter(self, nb_mois: int) -> float:
        """Projette le solde sur nb_mois avec intérêts composés mensuels."""
        t = self.taux / 100 / 12
        s = self.solde
        v = self.versements_permanents
        for _ in range(nb_mois):
            s = s * (1 + t) + v
        return round(s, 2)

    
    def __repr__(self):
        return f"Epargne({self.nom}, {self.solde} EUR, {self.taux}%/an)"


# Démonstration
import math

produits = [
    Epargne(1,  8000,  200, 3.0), # Livret A Vincent
    Epargne(9,  45000, 500, 4.5), # AV Rémy
    Epargne(14, 48000, 500, 6.0), # PEA Antoine
    Epargne(16, 80000, 1000, 7.0)] # Compte titres Emma


print(f"{'Produit':<20} {'Solde initial':>15} {'1 an':>12} {'3 ans':>12} {'10 ans':>12}")
for e in produits:
    print(f"{e.nom:<20} {e.solde:>13.0f} EUR"
          f" {e.projeter(12):>10.0f} EUR"
          f" {e.projeter(36):>10.0f} EUR"
          f" {e.projeter(120):>10.0f} EUR")

print()
print("=== Validation des plafonds ===")
try:
    Epargne(1, 25000, 0, 3.0)# Livret A au-delà du plafond 22 950 EUR
except ValueError as e:
    print(f"Plafond Livret A respecté — Erreur : {e}") 

"""
Attention, dans l'appli finale, les plafonds ont été enlevé car un livret réglementé peut aller 
au dessus de son plafonds avec les intérêts générés !
"""


try:
    Epargne(1, 5, 0, 3.0)  # Sous le minimum 10 EUR
except ValueError as e:
    print(f"Minimum Livret A respecté — Erreur : {e}")


Produit                Solde initial         1 an        3 ans       10 ans
Livret A                      8000 EUR      10677 EUR      16277 EUR      38743 EUR
Assurance Vie                45000 EUR      53193 EUR      70724 EUR     146114 EUR
PEA                          48000 EUR      57128 EUR      77109 EUR     169271 EUR
Compte titres                80000 EUR      98176 EUR     138564 EUR     333858 EUR

=== Validation des plafonds ===
Plafond Livret A respecté — Erreur : Plafond dépassé (22950 EUR)
Minimum Livret A respecté — Erreur : Solde sous le minimum requis (10 EUR)


## 5. La classe Client — Agrégation et orchestration

`Client` a le **rôle central** du modèle. Il agrège toutes les autres entités
et fait les calculs via plusieurs méthodes.

C'est une classe qui contient des listes d'autres objets et fournit des méthodes de synthèse.

```
Client
- List[Revenu]
- List[Depense]  
- List[Credit]
- List[Epargne]
- List[Compte]
- List[Patrimoine]
```


In [13]:
class Client:
    """
    Agrège toutes les entités financières d'une personne.
    
    Règle importante : un seul revenu principal à la fois (géré automatiquement
    par ajouter_revenu via désactivation des anciens principaux).
    """

    def __init__(self, nom: str):
        self.nom = nom
        self.revenus: List[Revenu] = []
        self.depenses: List[Depense] = []
        self.credits: List[Credit] = []
        self.epargnes: List[Epargne] = []

    def ajouter_revenu(self, revenu: Revenu):
        # Règle métier : un seul revenu principal
        if revenu.est_revenu_principal:
            for r in self.revenus:
                r.est_revenu_principal = False
        self.revenus.append(revenu)

    def ajouter_depense(self, depense: Depense):
        self.depenses.append(depense)

    def attacher_credit(self, credit: Credit):
        """Attache un crédit dont les emprunteurs sont déjà configurés."""
        self.credits.append(credit)

    def ajouter_epargne(self, epargne: Epargne):
        self.epargnes.append(epargne)

    
    # Calculs budget mensuel

    def revenus_du_mois(self, mois: int) -> float:
        return sum(r.montant_pour_mois(mois) for r in self.revenus)

    def depenses_du_mois(self, mois: int) -> dict:
        """Retourne les dépenses par type pour un mois."""
        fixes = variables = uniques = 0.0
        for d in self.depenses:
            if not d.est_a_appliquer(mois):
                continue
            if d.type_depense == "fixe":
                fixes += d.montant
            elif d.type_depense == "variable_mensuelle":
                variables += d.montant
            else:
                uniques += d.montant
        return {"fixes": fixes, "variables": variables, "uniques": uniques,
                "total": fixes + variables + uniques}

    def mensualites_credits(self) -> float:
        """Somme des mensualités pondérées par la part du client."""
        return sum(cr.mensualite_client(self) for cr in self.credits
                   if self in cr.emprunteur)

    def solde_mensuel(self, mois: int) -> float:
        revenus = self.revenus_du_mois(mois)
        charges = self.depenses_du_mois(mois)["total"] + self.mensualites_credits()
        return round(revenus - charges, 2)

    # Taux d'endettement bancaire

    PONDERATION = {
        "Salaire": 1.00, "Prime fixe": 0.50, "Bonus": 0.00,
        "Revenu non salarié": 1.00, "Revenu locatif": 0.70, "Revenu SCPI": 0.70,
        "Intérêt d'épargne": 0.50, "Retraite": 1.00,
        "Allocations familiales": 0.50,
    }

    def revenus_ponderes(self) -> float:
        """Revenus mensuels pondérés selon la grille bancaire."""
        total = 0.0
        for r in self.revenus:
            coeff = self.PONDERATION.get(r.type_de_revenu, 0.0)
            if coeff == 0.0:
                continue
            if r.periodicite == "mensuelle":
                total += r.montant * coeff
            elif r.periodicite == "annuelle":
                total += (r.montant / 12) * coeff
        return round(total, 2)

    def charges_bancaires(self) -> float:
        """Charges retenues par la banque (loyer + impôts + crédits)."""
        charges = 0.0
        for d in self.depenses:
            if d.categorie_depense in ("Loyer", "Impôts et taxes"):
                charges += d.montant
        charges += self.mensualites_credits()
        return round(charges, 2)

    def taux_endettement(self) -> float:
        rev = self.revenus_ponderes()
        if rev == 0:
            return 0.0
        return round(self.charges_bancaires() / rev * 100, 2)

    def __repr__(self):
        return (f"Client({self.nom} | {len(self.revenus)} revenus "
                f"| {len(self.depenses)} dépenses | {len(self.credits)} crédits)")


# Simulation complète : Alban K
albank = Client("Alban K")

salaire_albank = Revenu("Salaire", 2800, "mensuelle", jour=28)
salaire_albank.est_revenu_principal = True
albank.ajouter_revenu(salaire_albank)

albank.ajouter_depense(Depense("Loyer", 850, "fixe", "Loyer", jour=1))
albank.ajouter_depense(Depense("Assurance auto", 65, "fixe", "Assurance automobile", jour=5))
albank.ajouter_depense(Depense("Forfait mobile", 25, "fixe", "Autre (hors crédit, LLD, LOA)", jour=10))
albank.ajouter_depense(Depense("Alimentation", 400, "variable_mensuelle", "Alimentation", jour=15))
albank.ajouter_depense(Depense("Electricité", 80, "fixe", "Electricité", jour=20))

credit_conso = Credit(4, mensualite=180, jour_echeance=5, crd=4200, taux=5.5)
credit_conso.ajouter_emprunteur(albank, 1.0)
albank.attacher_credit(credit_conso)

print("=== Alban K — Budget mars ===")
print(albank)
print()

revenus_mars = albank.revenus_du_mois(3)
depenses_mars = albank.depenses_du_mois(3)
mensualites = albank.mensualites_credits()

print(f"Revenus du mois : {revenus_mars} EUR")
print(f"Dépenses fixes : {depenses_mars['fixes']} EUR")
print(f"Dépenses variables : {depenses_mars['variables']} EUR")
print(f"Total dépenses : {depenses_mars['total']} EUR")
print(f"Mensualités crédits : {mensualites} EUR")
print(f"Total charges : {depenses_mars['total'] + mensualites} EUR")
print(f"Solde mensuel : {albank.solde_mensuel(3)} EUR")
print()
print(f"Revenus pondérés bancaire: {albank.revenus_ponderes()} EUR")
print(f"Charges bancaires : {albank.charges_bancaires()} EUR")
print(f"Taux d'endettement : {albank.taux_endettement()} %")
print(f"Alerte > 35%: {'OUI' if albank.taux_endettement() > 35 else 'NON'}")


=== Alban K — Budget mars ===
Client(Alban K | 1 revenus | 5 dépenses | 1 crédits)

Revenus du mois : 2800.0 EUR
Dépenses fixes : 1020.0 EUR
Dépenses variables : 400.0 EUR
Total dépenses : 1420.0 EUR
Mensualités crédits : 180.0 EUR
Total charges : 1600.0 EUR
Solde mensuel : 1200.0 EUR

Revenus pondérés bancaire: 2800.0 EUR
Charges bancaires : 1030.0 EUR
Taux d'endettement : 36.79 %
Alerte > 35%: OUI


## 6. La classe Compte — Projection de solde temporelle

La projection de solde est un algorithme de **simulation temporelle** :
on parcourt mois par mois entre aujourd'hui et une date cible,
en appliquant les flux (revenus entrants, dépenses et crédits sortants).

**Points clés :**
- Le `Compte` ne stocke que le solde initial (source de vérité)
- Tous les flux sont calculés à la volée depuis les objets métier
- Supporte les **comptes joints** (agrégation multi-clients)


In [16]:
from datetime import datetime, timedelta

class Compte:
    """
    Compte bancaire avec projection de solde multi-clients.
    
    La projection est calculée à la demande depuis les objets Revenu/Depense/Credit.
    """

    def __init__(self, banque: str, nom_compte: str, solde_initial: float):
        self.banque = banque
        self.nom_compte = nom_compte
        self.solde_initial = float(solde_initial)
        self.client_ids: list = []

    @property
    def est_joint(self): return len(self.client_ids) > 1

    def projeter_solde(self, clients: list, aujourd_hui: datetime,
                       date_cible: datetime) -> float:
        """
        Projette le solde à date_cible en simulant les flux mois par mois.
        Agrège les flux de tous les clients passés en paramètre (compte joint).
        """
        if date_cible <= aujourd_hui:
            return self.solde_initial

        solde = self.solde_initial
        mois = aujourd_hui.month
        annee = aujourd_hui.year

        while True:
            if datetime(annee, mois, 1) > date_cible:
                break

            for client in clients:
                # Revenus
                for r in client.revenus:
                    montant = r.montant_pour_mois(mois)
                    if montant > 0:
                        jour_securise = min(r.jour, calendar.monthrange(annee, mois)[1])
                        date_flux = datetime(annee, mois, jour_securise)
                        if aujourd_hui < date_flux <= date_cible:
                            solde += montant

                # Dépenses
                for d in client.depenses:
                    if d.est_a_appliquer(mois):
                        mois_dep = d.mois if d.type_depense == "unique" else mois
                        jour_sec = min(d.jour, calendar.monthrange(annee, mois_dep)[1])
                        date_flux = datetime(annee, mois_dep, jour_sec)
                        if aujourd_hui < date_flux <= date_cible:
                            solde -= d.montant

                # Crédits (pondérés par la part du client)
                for cr in client.credits:
                    if client in cr.emprunteur:
                        mensualite = cr.mensualite_client(client)
                        jour_sec = min(cr.jour_echeance, calendar.monthrange(annee, mois)[1])
                        date_flux = datetime(annee, mois, jour_sec)
                        if aujourd_hui < date_flux <= date_cible:
                            solde -= mensualite

            mois = 1 if mois == 12 else mois + 1
            if mois == 1: annee += 1

        return round(solde, 2)


# Simulation : compte d'Alban
compte_albank = Compte("LCL", "Perso Alban", 3200)
albank.epargnes.append(Epargne(1, 8000, 200, 3.0))  # Livret A

aujourd_hui = datetime(2025, 3, 15)  # 15 mars 2025

# Fin de mois
dernier_jour = datetime(2025, 3, 31, 23, 59, 59)
solde_fdm = compte_marie.projeter_solde([albank], aujourd_hui, dernier_jour)

# Dans 3 mois
dans_3_mois = datetime(2025, 6, 30, 23, 59, 59)
solde_3m = compte_albank.projeter_solde([albank], aujourd_hui, dans_3_mois)

print("=== Compte Alban — Projections (au 15 mars 2025) ===")
print(f"Solde actuel: {compte_albank.solde_initial} EUR")
print(f"Solde fin mars : {solde_fdm} EUR")
print(f"Solde fin juin : {solde_3m} EUR")
print()
print("Explication fin mars :")
print(f"+ Salaire j28 : +2 800 EUR")
print(f"- Alimentation j15 : -400 EUR")
print(f"- Electricité j20 : -80 EUR")
print(f"- Crédit conso j5 : déjà passé le 15, non compté")
print(f"Net attendu : 3 200 + 2 800 - 400 - 80 = 5 520 EUR")
print(f"Calculé : {solde_fdm} EUR")


=== Compte Alban — Projections (au 15 mars 2025) ===
Solde actuel: 3200.0 EUR
Solde fin mars : 5920.0 EUR
Solde fin juin : 9520.0 EUR

Explication fin mars :
+ Salaire j28 : +2 800 EUR
- Alimentation j15 : -400 EUR
- Electricité j20 : -80 EUR
- Crédit conso j5 : déjà passé le 15, non compté
Net attendu : 3 200 + 2 800 - 400 - 80 = 5 520 EUR
Calculé : 5920.0 EUR


## 7. La classe Patrimoine — Actifs, passifs et rendement

Le patrimoine combine **plusieurs crédits** et **plusieurs revenus** associés.
La `valeur_detention` reflète la part réelle détenue (indivision).

**Indicateurs calculés :**
- `valeur_detention` = valeur × part%
- `patrimoine_net` = valeur_detention − CRD × part_emprunteur
- `rendement_brut` = revenus_annuels / valeur_détenue × 100
- `effort_immobilier` = mensualité_credit − loyer_mensuel


In [20]:
class Patrimoine:
    """
    Modélise un actif patrimonial avec gestion de l'indivision,
    de plusieurs crédits et de plusieurs revenus associés.
    """

    TYPE_PATRIMOINE = (
        "Résidence principale", "Résidence secondaire",
        "Immobilier locatif", "Terrain", "Parking/Garage",
        "Parts de SCI", "Biens d'usage (bijoux, voiture, oeuvres d'art)", "Autre"
    )

    def __init__(self, type_patrimoine: str, nom: str, valeur: float,
                 part: float = 100, credits: list = None, revenus: list = None):
        if type_patrimoine not in self.TYPE_PATRIMOINE:
            raise ValueError("Type de patrimoine invalide")
        if valeur < 0 or not (0 < part <= 100):
            raise ValueError("Valeur ou part invalide")
        
        self.type_patrimoine = type_patrimoine
        self.nom = nom
        self.valeur = float(valeur)
        self.part = float(part)
        self.credits = credits or []
        self.revenus = revenus or []

    @property
    def valeur_detention(self) -> float:
        return self.valeur * (self.part / 100)

    def crd_client(self, client) -> float:
        """CRD total pondéré par la part du client sur chaque crédit."""
        total = 0.0
        for cr in self.credits:
            if cr.crd is not None and client in cr.emprunteur:
                total += cr.crd * cr.emprunteur[client]
        return round(total, 2)

    def valeur_nette(self, client) -> float:
        return round(self.valeur_detention - self.crd_client(client), 2)

    def revenus_annuels(self) -> float:
        total = 0.0
        for r in self.revenus:
            if r.periodicite == "mensuelle":
                total += r.montant * 12
            elif r.periodicite == "annuelle":
                total += r.montant
        return total

    def rendement_brut(self) -> float:
        val = self.valeur_detention
        rev = self.revenus_annuels()
        return round(rev / val * 100, 2) if val > 0 and rev > 0 else 0.0

    def effort_immobilier(self, client) -> float:
        """Mensualités crédits - loyer mensuel perçu (positif = effort net)."""
        mensualites = sum(cr.mensualite_client(client) for cr in self.credits
                          if client in cr.emprunteur)
        loyer_mensuel = self.revenus_annuels() / 12
        return round(mensualites - loyer_mensuel, 2)

    def __repr__(self):
        return (f"Patrimoine({self.nom}, {self.valeur_detention:.0f} EUR détenu "
                f"[{self.part:.0f}%])")


# Simulation : Pierre
pierre = Client("Pierre")
rev_salaire_pierre = Revenu("Salaire", 4100, "mensuelle", jour=28)
rev_loyer_pierre = Revenu("Revenu locatif", 750, "mensuelle", jour=5)
pierre.ajouter_revenu(rev_salaire_pierre)
pierre.ajouter_revenu(rev_loyer_pierre)

# Crédit RP (50/50 avec Sophie)
sophie = Client("Sophie")
credit_rp = Credit(1, mensualite=1200, jour_echeance=1, crd=185000, taux=2.1)
credit_rp.ajouter_emprunteur(sophie, 0.5)
credit_rp.ajouter_emprunteur(pierre, 0.5)
pierre.attacher_credit(credit_rp)
sophie.attacher_credit(credit_rp)

# Crédit locatif (Pierre 100%)
credit_locatif_pierre = Credit(1, mensualite=680, jour_echeance=15, crd=95000, taux=3.2)
credit_locatif_pierre.ajouter_emprunteur(pierre, 1.0)
pierre.attacher_credit(credit_locatif_pierre)

# Patrimoines
rp = Patrimoine("Résidence principale", "Maison principale", 320000,
                part=50, credits=[credit_rp])

locatif = Patrimoine("Immobilier locatif", "Appartement locatif", 180000,
                     part=100, credits=[credit_locatif_pierre], revenus=[rev_loyer_pierre])

print("=== Pierre — Analyse patrimoniale ===")
print()
print(f"--- {rp} ---")
print(f"Valeur détenue (50%) : {rp.valeur_detention:,.0f} EUR")
print(f"CRD part Pierre : {rp.crd_client(pierre):,.0f} EUR")
print(f"Valeur nette : {rp.valeur_nette(pierre):,.0f} EUR")
print(f"Effort immo RP : {rp.effort_immobilier(pierre):,.0f} EUR/mois")
print()
print(f"--- {locatif} ---")
print(f"Valeur détenue (100%) : {locatif.valeur_detention:,.0f} EUR")
print(f"CRD Pierre : {locatif.crd_client(pierre):,.0f} EUR")
print(f"Valeur nette : {locatif.valeur_nette(pierre):,.0f} EUR")
print(f"Revenus locatifs/an : {locatif.revenus_annuels():,.0f} EUR")
print(f"Rendement brut : {locatif.rendement_brut():.2f} %")
print(f"Effort immo locatif : {locatif.effort_immobilier(pierre):,.0f} EUR/mois")
print(f" (négatif = bien autofinancé)")
print()

# Récap Pierre
patrimoine_brut = rp.valeur_detention + locatif.valeur_detention
patrimoine_net = rp.valeur_nette(pierre) + locatif.valeur_nette(pierre)
print(f"Patrimoine brut Pierre : {patrimoine_brut:,.0f} EUR")
print(f"Patrimoine net Pierre : {patrimoine_net:,.0f} EUR")
print()
print(f"Taux d'endettement Pierre : {pierre.taux_endettement()} %")


=== Pierre — Analyse patrimoniale ===

--- Patrimoine(Maison principale, 160000 EUR détenu [50%]) ---
Valeur détenue (50%) : 160,000 EUR
CRD part Pierre : 92,500 EUR
Valeur nette : 67,500 EUR
Effort immo RP : 600 EUR/mois

--- Patrimoine(Appartement locatif, 180000 EUR détenu [100%]) ---
Valeur détenue (100%) : 180,000 EUR
CRD Pierre : 95,000 EUR
Valeur nette : 85,000 EUR
Revenus locatifs/an : 9,000 EUR
Rendement brut : 5.00 %
Effort immo locatif : -70 EUR/mois
 (négatif = bien autofinancé)

Patrimoine brut Pierre : 340,000 EUR
Patrimoine net Pierre : 152,500 EUR

Taux d'endettement Pierre : 27.68 %


## 8. Récapitulatif — Principes POO appliqués

### Principes respectés dans BudgetPylot

`Revenu` gère les revenus, `Credit` les crédits, `Client` orchestre le tout. Quand on ajoute donc un élément object (par exemple quand on ajoute un revenu), on modifie un tuple et non la logique.
Les classes sont cohérentes et interchangeables dans les listes.

### Patterns utilisés

```python
# Pattern pour l'agrégation — Client contient et orchestre
class Client:
    revenus: List[Revenu]
    credits: List[Credit]
    def solde_mensuel(self, mois): ... 

# Credit <->Client avec part (chaque client détenant un crédit à un pourcentage de détention sur ce crédit
class Credit:
    emprunteur: dict[Client, float]  # {client: part}
    def mensualite_client(self, client): return self.mensualite * self.emprunteur[client]

# Types valides comme attributs de classe
class Revenu:
    TYPE_REVENU = ("salaire", "prime fixe", ...)  # immutable, partagé
    
# Periodicite
def montant_pour_mois(self, mois):
    if self.periodicite == "mensuelle": return self.montant
    elif self.mois == mois: return self.montant
    return 0.0
```

### Architecture Flask complète

```
models/ # Entités POO (ce notebook)
├── client.py  # Agrégateur central
├── revenu.py  # Règles périodicité
├── credit.py  # Co-emprunteurs + parts
├── epargne.py  # Plafonds réglementaires
└── patrimoine.py # Indivision + rendement

services/
├── statistiques.py  # Calculs budget/endettement/épargne/patrimoine
└── taux_manager.py  # API Banque de France + fallback

data/
└── csv_manager.py  # Sérialisation/désérialisation

app.py  # Routes Flask + isolation sessions utilisateurs
```

---

*BudgetPylot est un projet personnel développé en Python/Flask avec une architecture
POO complète. Chaque classe modélise un concept financier réel avec ses règles métier. Le code complet est disponible sur github.*
